<!-- WARNING: THIS FILE WAS AUTOGENERATED! DO NOT EDIT! -->

In [0]:
#| echo: false
#| output: asis
show_doc(TemporalMeanPoolClassifier)

---

[source](https://github.com/benmfox/PhysioJEPA/blob/main/physiojepa/supervised_patchtst.py#L14){target="_blank" style="float:right; font-size:smaller"}

### TemporalMeanPoolClassifier

```python
def TemporalMeanPoolClassifier(
    d_model, c_in, num_classes:int=1
):
```

*Mean-pool contextualized patches per channel before classification.*

In [0]:
#| echo: false
#| output: asis
show_doc(SupervisedPatchTST)

---

[source](https://github.com/benmfox/PhysioJEPA/blob/main/physiojepa/supervised_patchtst.py#L27){target="_blank" style="float:right; font-size:smaller"}

### SupervisedPatchTST

```python
def SupervisedPatchTST(
    c_in, patch_size, patch_stride, num_patches, d_model, n_heads, d_ff, num_layers, augmentations:NoneType=None,
    mask_ratio:float=0.0, shared_embedding:bool=False, dropout:float=0.0, attn_dropout:float=0.0, act:str='gelu',
    pre_norm:bool=False, pe_type:str='tAPE', qkv_bias:bool=True, init_std:float=0.02, tokenizer_type:str='simple',
    tokenizer_kwargs:NoneType=None, classifier_mlp_ratio:float=4.0, classifier_depth:int=1,
    classifier_init_std:float=0.02, classifier_qkv_bias:bool=True, classifier_complete_block:bool=True,
    classifier_affine:bool=False, classifier_type:str='attentive', num_classes:int=1
):
```

*Standalone PatchTST encoder with an end-to-end supervised head.*

In [ ]:
# Standard supervised PatchTST: rotary positional encoding with a linear tokenizer and the default attentive classifier.
# This general-purpose configuration checks output shape, encoder/classifier gradients, and head separation.
torch.manual_seed(16)
model = SupervisedPatchTST(
    c_in=3,
    patch_size=4,
    patch_stride=4,
    num_patches=8,
    d_model=16,
    n_heads=4,
    d_ff=32,
    num_layers=2,
    shared_embedding=False,
    pe_type='rotary',
    tokenizer_type='linear',
    num_classes=1,
)
x = torch.randn(2, 3, 32)
target = torch.tensor([[0.0], [1.0]])
logits = model(x)
assert logits.shape == (2, 1)
assert not hasattr(model.encoder, 'head')
nn.BCEWithLogitsLoss()(logits, target).backward()
assert any(p.grad is not None for p in model.encoder.parameters())
assert any(p.grad is not None for p in model.classifier.parameters())

# Compact FCN-comparison model: 1-second patches, simple tokenization, and mean pooling.
# Its width/depth are chosen to match the 260,281-parameter FCN baseline, not to maximize capacity.
compact_model = SupervisedPatchTST(
    c_in=3,
    patch_size=125,
    patch_stride=125,
    num_patches=1800,
    d_model=96,
    n_heads=4,
    d_ff=384,
    num_layers=2,
    shared_embedding=False,
    pe_type='rotary',
    tokenizer_type='simple',
    classifier_type='mean',
    num_classes=1,
)
assert sum(p.numel() for p in compact_model.parameters()) == 260_281
# Minimal mean-pooling variant: a small shape-only check for the same classifier interface.
mean_model = SupervisedPatchTST(
    c_in=3,
    patch_size=4,
    patch_stride=4,
    num_patches=8,
    d_model=16,
    n_heads=4,
    d_ff=32,
    num_layers=1,
    shared_embedding=False,
    pe_type='rotary',
    tokenizer_type='linear',
    classifier_type='mean',
    num_classes=1,
)
mean_logits = mean_model(torch.randn(2, 3, 32))
assert mean_logits.shape == (2, 1)

# Configuration guard: unsupported classifier names must fail early rather than silently selecting another head.
try:
    SupervisedPatchTST(
        c_in=3,
        patch_size=4,
        patch_stride=4,
        num_patches=8,
        d_model=16,
        n_heads=4,
        d_ff=32,
        num_layers=1,
        classifier_type='unknown',
    )
except ValueError as exc:
    assert "classifier_type" in str(exc)
else:
    raise AssertionError("invalid classifier_type should raise ValueError")